In [11]:
import pandas as pd
import rasterio
import re

path_raster = "data/geollm/ppp_2020_1km_Aggregated.tif"
path_prompts = "data/geollm/world_prompts.jsonl"    

In [12]:
geollm = pd.read_json(path_prompts, lines=True)
geollm.shape

(2000, 1)

In [13]:
print(geollm.iloc[0]["text"])

Coordinates: (-5.87958, 14.98625)

Address: "Uíge, Angola"

Nearby Places:
"
3.4 km North-East: Kimbala
4.7 km North-East: Kimpangu
4.8 km North-East: Kimvindu
5.9 km North-East: Kikongo
15.7 km North-West: Tunda
15.9 km North-East: Kungu Lusanga
17.5 km North: Luzanga
18.2 km North: Mbanza Ngombe
18.4 km North-West: Wene
18.6 km North-West: Kongo Dia Kati
"

<TASK> (On a Scale from 0.0 to 9.9): 


In [ ]:
def extract_coord_from_line(text):
    pattern = r'Coordinates:\s*\((-?\d+\.\d+),\s*(-?\d+\.\d+)\)'

    match = re.search(pattern, text)
    if match:
        lon = float(match.group(1))
        lat = float(match.group(2))
    
    return lon, lat

def get_population_at(src, lon, lat):
    return list(src.sample([(lon, lat)]))[0][0]

def get_coord_from_json(json_file, line_range=(0, 10)):
    prompts = []
    lons = []
    lats = []
    df_json = pd.read_json(json_file, lines=True)
    for i in range(*line_range):
        prompt = df_json.iloc[i]['text']
        lon, lat = extract_coord_from_line(prompt)

        prompts.append(prompt)
        lons.append(lon)
        lats.append(lat)
    
    df_ret = pd.DataFrame({"prompt" : prompts, "lon": lons, "lat" : lats})
    return df_ret

In [21]:
df = get_coord_from_json(path_prompts, line_range=(0,2000))

In [ ]:
print(df.shape)
df.head()

(2000, 3)


,prompt,lon,lat
0,"Coordinates: (-5.87958, 14.98625)\n\nAddress: ...",-5.87958,14.98625
1,"Coordinates: (67.29542, 14.72792)\n\nAddress: ...",67.29542,14.72792
2,"Coordinates: (58.37875, -134.63042)\n\nAddress...",58.37875,-134.63042
3,"Coordinates: (55.05375, 58.97792)\n\nAddress: ...",55.05375,58.97792
4,"Coordinates: (53.18708, 8.23625)\n\nAddress: ""...",53.18708,8.23625


In [7]:
lon = -5.87958
lat = 14.98625

coord1 = (lon,lat)
print(*coord1)

-5.87958 14.98625


In [ ]:
src = rasterio.open(path_raster)

print(get_population_at(src, *coord1))

src.close()

8.069204


In [ ]:
with rasterio.open(path_raster) as src:
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bands:", src.count)
    print("CRS:", src.crs)
    
    band1 = src.read(1)
print("Min:", band1.min())
print("Max:", band1.max())
print("Mean:", band1.mean())

Width: 43200
Height: 18720
Bands: 1
CRS: EPSG:4326
